# Session 19 Exercise: Unsupervised Learning

Cluster campaign customers using only pre-response features.


In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
campaign = pd.read_csv(DATA / "campaign_response.csv")
features = ["age", "visits_last_month", "emails_opened", "discount_pct", "prior_spend_eur", "segment"]
numeric = ["age", "visits_last_month", "emails_opened", "discount_pct", "prior_spend_eur"]
preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["segment"]),
    ]
)
X = preprocess.fit_transform(campaign[features])


In [ ]:
rows = []
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    rows.append({"k": k, "silhouette": silhouette_score(X, labels)})
pd.DataFrame(rows)


In [ ]:
labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)
campaign.assign(cluster=labels).groupby("cluster")[numeric + ["responded"]].mean().round(2)


Name each cluster using the summary table. Do not use the response rate as an input to the clustering step.
